# LeetCode 637 — Average of Levels in Binary Tree bằng Python

**Link đề:** https://leetcode.com/problems/average-of-levels-in-binary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def averageOfLevels(self, root: TreeNode | None) -> list[float]:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **tầng (level)** của cây nhị phân và cách LeetCode ghi cây theo tầng.
2. Thành thạo **BFS (duyệt theo chiều rộng)** bằng `collections.deque`, đặc biệt mẹo **chụp kích thước tầng** `size = len(queue)`.
3. Viết 4 lời giải: BFS với queue → BFS với list "tầng hiện tại/tầng kế" → DFS đệ quy mang theo `depth` → DFS bằng stack.
4. Hiểu phép chia `/` (ra `float`) vs `//`, sai số dấu phẩy động và yêu cầu "sai lệch ≤ `10^-5`".
5. Trace từng tầng, chứng minh, Time/Space, 6 bài tập "theo tầng" có gợi ý và đáp án cuối.

## 1. Đề bài và ví dụ

Cho `root` của cây nhị phân, trả về **giá trị trung bình** của các node **trên mỗi tầng**, dạng list (tầng 0 trước). Đáp án sai lệch trong `10^-5` so với giá trị đúng vẫn được chấp nhận.

Ví dụ 1: `root = [3,9,20,null,null,15,7]` → `[3.00000, 14.50000, 11.00000]`

```text
        3          tầng 0: 3            -> 3
       / \
      9   20       tầng 1: 9, 20        -> (9 + 20) / 2 = 14.5
         /  \
        15   7     tầng 2: 15, 7        -> (15 + 7) / 2 = 11
```

Ví dụ 2: `root = [3,9,20,15,7]` → `[3.00000, 14.50000, 11.00000]` (tầng 2 là `15, 7` nhưng nằm dưới node `9`).

**Constraints:**
- Số node trong `[1, 10^4]` (cây **không rỗng**).
- `-2^31 <= Node.val <= 2^31 - 1`.

Lưu ý: tổng một tầng có thể vượt giới hạn số nguyên 32-bit (ví dụ hai node `2^31 − 1`). Python không lo tràn số, nhưng ở Java/C++ phải dùng `long`/`double`.

## 2. Nền tảng: `TreeNode`, dựng cây, in cây theo tầng

Mỗi node có `val`, `left`, `right`. **Tầng (level/depth)** của node = số cạnh từ gốc xuống node đó (gốc ở tầng 0).

Ta tự định nghĩa `TreeNode` + `build_tree` (từ list LeetCode, dùng queue) để notebook chạy độc lập.

In [1]:
from collections import deque
import random, math


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def show_tree(root, prefix="", label="root"):
    if root is None:
        return
    print(f"{prefix}{label}: {root.val}")
    show_tree(root.left, prefix + "    ", "L")
    show_tree(root.right, prefix + "    ", "R")


EX1 = [3, 9, 20, None, None, 15, 7]
EX2 = [3, 9, 20, 15, 7]
show_tree(build_tree(EX1))
print("---")
show_tree(build_tree(EX2))

root: 3
    L: 9
    R: 20
        L: 15
        R: 7
---
root: 3
    L: 9
        L: 15
        R: 7
    R: 20


Thử tính tầng của từng node bằng một hàm đệ quy nhỏ: con có tầng = tầng cha + 1.

In [2]:
def list_depths(node, depth=0):
    if node is None:
        return []
    return [(node.val, depth)] + list_depths(node.left, depth + 1) + list_depths(node.right, depth + 1)

print(list_depths(build_tree(EX1)))   # (giá trị, tầng)

[(3, 0), (9, 1), (20, 1), (15, 2), (7, 2)]


### 2.1. Queue (`deque`) – vào trước ra trước
BFS cần xử lý node **theo thứ tự được phát hiện**: gốc, rồi các con của gốc, rồi các cháu… Đó đúng là **queue** (FIFO). `deque.append` thêm cuối, `deque.popleft` lấy đầu – đều `O(1)`. (Dùng `list.pop(0)` cũng được nhưng `O(n)` mỗi lần.)

In [3]:
q = deque()
for name in ["gốc", "con trái", "con phải"]:
    q.append(name)
print("queue:", list(q))
print("popleft:", q.popleft(), "| còn:", list(q))

queue: ['gốc', 'con trái', 'con phải']
popleft: gốc | còn: ['con trái', 'con phải']


### 2.2. Phép chia `/` và `//`, sai số `float`
- `/` luôn trả `float`: `29 / 2 = 14.5`, `6 / 2 = 3.0`.
- `//` chia lấy phần nguyên (làm tròn xuống): `29 // 2 = 14` – **sai** cho trung bình.
- `float` có sai số nhỏ (ví dụ `0.1 + 0.2`), nên khi test so sánh với sai lệch cho phép bằng `math.isclose` hoặc `abs(a - b) <= 1e-5`.

In [4]:
print("29 / 2  =", 29 / 2, "| 29 // 2 =", 29 // 2, "| 6 / 2 =", 6 / 2)
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3, "| isclose:", math.isclose(0.1 + 0.2, 0.3))

29 / 2  = 14.5 | 29 // 2 = 14 | 6 / 2 = 3.0
0.1 + 0.2 == 0.3 ? False | isclose: True


## 3. Cách nghĩ trực giác: gom node theo tầng rồi tính trung bình

Bằng tay: đi từng tầng, liệt kê giá trị, cộng lại, chia cho số node. Câu hỏi kỹ thuật: **làm sao biết node nào thuộc tầng nào?**

Hai hướng:
- **BFS:** duyệt theo tầng – tự nhiên biết tầng.
- **DFS:** duyệt sâu, nhưng **mang theo `depth`** khi gọi xuống con; cộng dồn vào `sums[depth]`, `counts[depth]`.

Baseline dễ hiểu nhất: dùng DFS gom **toàn bộ giá trị** theo tầng vào `dict` `depth -> list giá trị`, rồi tính trung bình. Tốn thêm bộ nhớ lưu mọi giá trị, nhưng rất rõ ràng.

In [5]:
def average_levels_collect(root):
    levels = {}                        # depth -> list các giá trị

    def dfs(node, depth):
        if node is None:
            return
        levels.setdefault(depth, []).append(node.val)
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    print("Gom theo tầng:", levels)
    return [sum(levels[d]) / len(levels[d]) for d in range(len(levels))]

print(average_levels_collect(build_tree(EX1)))

Gom theo tầng: {0: [3], 1: [9, 20], 2: [15, 7]}
[3.0, 14.5, 11.0]


`levels.setdefault(depth, [])` trả về list của `depth`, **tạo list rỗng nếu chưa có** – rồi `.append` vào đó.

**Nút thắt:** lưu cả `n` giá trị chỉ để cộng – thừa. Ta chỉ cần **tổng** và **số lượng** mỗi tầng. Và với BFS ta còn không cần lưu theo tầng: xử lý xong tầng nào tính trung bình tầng đó.

## 4. Insight BFS: chụp `size = len(queue)` ở đầu mỗi tầng

Khi bắt đầu một tầng, queue chứa **đúng và chỉ** các node của tầng đó. Ghi lại `size = len(queue)`, rồi `popleft` đúng `size` lần: đó là toàn bộ tầng hiện tại; các con được `append` vào cuối queue thuộc tầng **kế tiếp** và không bị lẫn vào vì ta chỉ lấy `size` phần tử.

```text
queue = [3]            -> size=1: lấy 3, thêm 9, 20      -> trung bình 3
queue = [9, 20]        -> size=2: lấy 9, 20; thêm 15, 7  -> trung bình 14.5
queue = [15, 7]        -> size=2: lấy 15, 7              -> trung bình 11
queue = []             -> dừng
```

## 5. Cách 1 — BFS với queue: giảng từng đoạn rồi ghép

1. `queue = deque([root])`, `result = []`.
2. `while queue:` – còn tầng chưa xử lý.
3. `size = len(queue)`, `level_sum = 0`.
4. Lặp `size` lần: `node = queue.popleft()`, cộng `node.val`, thêm con trái/phải (nếu có).
5. `result.append(level_sum / size)`.

In [6]:
def average_levels_bfs(root):
    if root is None:
        return []
    queue = deque([root])
    result = []
    while queue:
        size = len(queue)                  # số node của tầng hiện tại
        level_sum = 0
        for _ in range(size):
            node = queue.popleft()
            level_sum += node.val
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level_sum / size)
    return result


print(average_levels_bfs(build_tree(EX1)))
print(average_levels_bfs(build_tree(EX2)))

[3.0, 14.5, 11.0]
[3.0, 14.5, 11.0]


`for _ in range(size)`: dấu `_` là quy ước "biến không dùng tới" – ta chỉ cần lặp đúng `size` lần.

### 5.1. TRACE BFS

| Tầng | queue đầu tầng | size | node lấy ra | con thêm vào | tổng | trung bình |
|---:|---|---:|---|---|---:|---:|
| 0 | `[3]` | 1 | 3 | 9, 20 | 3 | 3.0 |
| 1 | `[9, 20]` | 2 | 9, 20 | 15, 7 (con của 20) | 29 | 14.5 |
| 2 | `[15, 7]` | 2 | 15, 7 | – | 22 | 11.0 |

In [7]:
def trace_bfs(root):
    queue = deque([root])
    result, level = [], 0
    while queue:
        size = len(queue)
        print(f"Tầng {level}: queue đầu tầng = {[n.val for n in queue]}, size = {size}")
        level_sum = 0
        for _ in range(size):
            node = queue.popleft()
            level_sum += node.val
            kids = [c.val for c in (node.left, node.right) if c]
            queue.extend(c for c in (node.left, node.right) if c)
            print(f"   lấy {node.val:>3}, thêm con {kids}, tổng tạm = {level_sum}")
        result.append(level_sum / size)
        print(f"   => trung bình tầng {level} = {level_sum} / {size} = {result[-1]}")
        level += 1
    return result

trace_bfs(build_tree(EX1))

Tầng 0: queue đầu tầng = [3], size = 1
   lấy   3, thêm con [9, 20], tổng tạm = 3
   => trung bình tầng 0 = 3 / 1 = 3.0
Tầng 1: queue đầu tầng = [9, 20], size = 2
   lấy   9, thêm con [], tổng tạm = 9
   lấy  20, thêm con [15, 7], tổng tạm = 29
   => trung bình tầng 1 = 29 / 2 = 14.5
Tầng 2: queue đầu tầng = [15, 7], size = 2
   lấy  15, thêm con [], tổng tạm = 15
   lấy   7, thêm con [], tổng tạm = 22
   => trung bình tầng 2 = 22 / 2 = 11.0


[3.0, 14.5, 11.0]

## 6. Cách 2 — BFS bằng hai list "tầng hiện tại" và "tầng kế"

Không cần `deque` hay `size`: giữ `level` là list các node tầng hiện tại; dựng `next_level` từ các con; gán `level = next_level`. Rất dễ đọc cho người mới.

In [8]:
def average_levels_two_lists(root):
    result = []
    level = [root] if root else []
    while level:
        result.append(sum(node.val for node in level) / len(level))
        next_level = []
        for node in level:
            if node.left:
                next_level.append(node.left)
            if node.right:
                next_level.append(node.right)
        level = next_level
    return result

print(average_levels_two_lists(build_tree(EX1)))
print(average_levels_two_lists(build_tree(EX2)))

[3.0, 14.5, 11.0]
[3.0, 14.5, 11.0]


`sum(node.val for node in level)` dùng **generator expression**: tính tổng mà không tạo list trung gian.

## 7. Cách 3 — DFS đệ quy mang theo `depth`

Giữ hai list `sums`, `counts` với chỉ số là tầng. Khi gặp tầng mới lần đầu (`depth == len(sums)`), `append` phần tử mới; ngược lại cộng dồn. Vì DFS luôn đi xuống từng tầng một (`depth` tăng 1 mỗi lần), tầng mới luôn có `depth` đúng bằng `len(sums)`.

In [9]:
def average_levels_dfs(root):
    sums, counts = [], []

    def dfs(node, depth):
        if node is None:
            return
        if depth == len(sums):            # lần đầu tới tầng này
            sums.append(0)
            counts.append(0)
        sums[depth] += node.val
        counts[depth] += 1
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return [s / c for s, c in zip(sums, counts)]

print(average_levels_dfs(build_tree(EX1)))
print(average_levels_dfs(build_tree(EX2)))

[3.0, 14.5, 11.0]
[3.0, 14.5, 11.0]


In [10]:
def trace_dfs(root):
    sums, counts = [], []

    def dfs(node, depth):
        if node is None:
            return
        if depth == len(sums):
            sums.append(0)
            counts.append(0)
        sums[depth] += node.val
        counts[depth] += 1
        print("  " * depth + f"thăm {node.val} ở tầng {depth} -> sums={sums}, counts={counts}")
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return [s / c for s, c in zip(sums, counts)]

print(trace_dfs(build_tree(EX2)))

thăm 3 ở tầng 0 -> sums=[3], counts=[1]
  thăm 9 ở tầng 1 -> sums=[3, 9], counts=[1, 1]
    thăm 15 ở tầng 2 -> sums=[3, 9, 15], counts=[1, 1, 1]
    thăm 7 ở tầng 2 -> sums=[3, 9, 22], counts=[1, 1, 2]
  thăm 20 ở tầng 1 -> sums=[3, 29, 22], counts=[1, 2, 2]
[3.0, 14.5, 11.0]


`zip(sums, counts)` ghép từng cặp `(sums[i], counts[i])`.

## 8. Cách 4 — DFS bằng stack `(node, depth)` (không đệ quy)

Cây có tới `10^4` node, nếu lệch hoàn toàn thì cao `10^4` – đệ quy có thể vượt giới hạn ~1000 của Python. Dùng stack tự quản lý.

In [11]:
def average_levels_dfs_stack(root):
    if root is None:
        return []
    sums, counts = [], []
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth == len(sums):
            sums.append(0)
            counts.append(0)
        sums[depth] += node.val
        counts[depth] += 1
        if node.right:
            stack.append((node.right, depth + 1))
        if node.left:
            stack.append((node.left, depth + 1))
    return [s / c for s, c in zip(sums, counts)]

def chain(n):
    root = cur = TreeNode(0)
    for v in range(1, n):
        cur.right = TreeNode(v)
        cur = cur.right
    return root

print(average_levels_dfs_stack(build_tree(EX1)))
deep = chain(10000)
print("Cây lệch 10000 node, 3 tầng cuối:", average_levels_dfs_stack(deep)[-3:])
try:
    average_levels_dfs(deep)
except RecursionError:
    print("DFS đệ quy: RecursionError trên cây lệch 10000 node")

[3.0, 14.5, 11.0]
Cây lệch 10000 node, 3 tầng cuối: [9997.0, 9998.0, 9999.0]
DFS đệ quy: RecursionError trên cây lệch 10000 node


Vẫn đúng điều kiện `depth == len(sums)` vì mọi node ở tầng `d` chỉ được đẩy vào stack sau khi cha (tầng `d − 1`) đã được pop – nên tầng `d − 1` chắc chắn đã có trong `sums` trước khi gặp tầng `d`.

## 9. Code nộp LeetCode

Bản chính: BFS với queue.

In [12]:
from collections import deque

class Solution:
    def averageOfLevels(self, root: TreeNode | None) -> list[float]:
        result = []
        queue = deque([root])
        while queue:
            size = len(queue)
            level_sum = 0
            for _ in range(size):
                node = queue.popleft()
                level_sum += node.val
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            result.append(level_sum / size)
        return result


def close_lists(a, b, tol=1e-5):
    return len(a) == len(b) and all(abs(x - y) <= tol for x, y in zip(a, b))

sol = Solution()
assert close_lists(sol.averageOfLevels(build_tree(EX1)), [3.0, 14.5, 11.0])
assert close_lists(sol.averageOfLevels(build_tree(EX2)), [3.0, 14.5, 11.0])
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


(Đề bảo đảm cây có ít nhất 1 node nên không cần kiểm tra `root is None`; các hàm demo ở trên vẫn xử lý rỗng cho an toàn.)

## 10. Vì sao đúng? (invariant)

**BFS – invariant:** ở đầu mỗi vòng `while`, `queue` chứa **đúng tất cả** node của tầng `d` (theo thứ tự trái → phải) và không node nào khác.
- **Khởi tạo:** `queue = [root]` = tầng 0.
- **Duy trì:** ta lấy đúng `size` node (cả tầng `d`) và thêm **mọi** con của chúng – mọi node tầng `d + 1` đều là con của một node tầng `d`, nên sau vòng này queue = đúng tầng `d + 1`.
- **Kết thúc:** queue rỗng ⇔ tầng kế tiếp không có node ⇔ đã qua tầng sâu nhất.
Mỗi vòng tính đúng `tổng / số node` của một tầng → `result[d]` đúng.

**DFS:** mỗi node được thăm đúng một lần với `depth` = tầng thật của nó (gốc 0, con = cha + 1), nên `sums[d]` và `counts[d]` là tổng và số node tầng `d`.

## 11. Time / Space Complexity và so sánh

`n` = số node, `h` = chiều cao, `w` = độ rộng lớn nhất (số node nhiều nhất trên một tầng).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 0. Gom mọi giá trị theo tầng | `O(n)` | `O(n)` | Dễ hiểu, tốn bộ nhớ |
| 1. BFS queue | `O(n)` | `O(w)` | Bản chính, tự nhiên nhất |
| 2. BFS hai list | `O(n)` | `O(w)` | Không cần `size` |
| 3. DFS đệ quy | `O(n)` | `O(h)` call stack + `O(h)` cho `sums` | Có thể `RecursionError` |
| 4. DFS stack | `O(n)` | `O(h)` – `O(n)` | An toàn với cây sâu |

Cây hoàn chỉnh: `w ≈ n/2` (tầng đáy) nên BFS tốn `O(n)` bộ nhớ; cây lệch: `w = 1` nhưng `h = n`.

Kiểm chứng Space của BFS: đo kích thước queue lớn nhất trên cây **hoàn chỉnh** (rộng) và cây **lệch** (hẹp).

In [13]:
def max_queue_size(root):
    queue, best = deque([root]), 1
    while queue:
        for _ in range(len(queue)):
            node = queue.popleft()
            queue.extend(c for c in (node.left, node.right) if c)
        best = max(best, len(queue))
    return best

complete = build_tree(list(range(1, 128)))      # cây hoàn chỉnh 127 node, 7 tầng
print("Cây hoàn chỉnh 127 node: queue lớn nhất =", max_queue_size(complete), "(= số node tầng đáy)")
print("Cây lệch 10000 node    : queue lớn nhất =", max_queue_size(deep))

Cây hoàn chỉnh 127 node: queue lớn nhất = 64 (= số node tầng đáy)
Cây lệch 10000 node    : queue lớn nhất = 1


## 12. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

In [14]:
methods = {
    "collect": lambda r: average_levels_collect_quiet(r),
    "bfs": average_levels_bfs,
    "two_lists": average_levels_two_lists,
    "dfs": average_levels_dfs,
    "dfs_stack": average_levels_dfs_stack,
    "Solution": Solution().averageOfLevels,
}

def average_levels_collect_quiet(root):
    levels = {}
    stack = [(root, 0)] if root else []
    while stack:
        node, d = stack.pop()
        levels.setdefault(d, []).append(node.val)
        stack += [(c, d + 1) for c in (node.left, node.right) if c]
    return [sum(levels[d]) / len(levels[d]) for d in range(len(levels))]

BIG = 2**31 - 1
cases = [
    ("đề 1", EX1, [3.0, 14.5, 11.0]),
    ("đề 2", EX2, [3.0, 14.5, 11.0]),
    ("1 node", [5], [5.0]),
    ("giá trị âm", [-1, -2, -3], [-1.0, -2.5]),
    ("tràn 32-bit", [BIG, BIG, BIG], [float(BIG), float(BIG)]),
    ("lệch trái", [1, 2, None, 3, None, 4], [1.0, 2.0, 3.0, 4.0]),
    ("tầng không đều", [1, 2, 3, 4, None, None, 5, 6], [1.0, 2.5, 4.5, 6.0]),
]
for name, values, expected in cases:
    for mname, f in methods.items():
        got = f(build_tree(values))
        assert close_lists(got, expected), (name, mname, got)
    print(f"{name:<16} PASS -> {expected}")

đề 1             PASS -> [3.0, 14.5, 11.0]
đề 2             PASS -> [3.0, 14.5, 11.0]
1 node           PASS -> [5.0]
giá trị âm       PASS -> [-1.0, -2.5]
tràn 32-bit      PASS -> [2147483647.0, 2147483647.0]
lệch trái        PASS -> [1.0, 2.0, 3.0, 4.0]
tầng không đều   PASS -> [1.0, 2.5, 4.5, 6.0]


In [15]:
def random_tree(n, rng):
    root = TreeNode(rng.randint(-100, 100))
    nodes = [root]
    while len(nodes) < n:
        parent = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(parent, side) is None:
            child = TreeNode(rng.randint(-100, 100))
            setattr(parent, side, child)
            nodes.append(child)
    return root

rng = random.Random(637)
for _ in range(300):
    root = random_tree(rng.randint(1, 40), rng)
    expected = average_levels_collect_quiet(root)
    for mname, f in methods.items():
        assert close_lists(f(root), expected), mname
print("300 cây ngẫu nhiên: 6 cách khớp nhau — PASS")

300 cây ngẫu nhiên: 6 cách khớp nhau — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Không chụp `size`, dùng `while queue` bên trong
Vòng trong sẽ ăn luôn các node tầng sau vừa được thêm → cả cây thành một "tầng".

### Lỗi 2: Dùng `len(queue)` **trong** điều kiện `range` sau khi đã thêm con
`range(len(queue))` được tính **một lần** khi bắt đầu `for` nên thực ra vẫn đúng; nhưng nếu dùng `while i < len(queue)` thì `len` thay đổi theo – sai.

### Lỗi 3: Chia `//` thay vì `/`
`29 // 2 = 14` thay vì `14.5`.

### Lỗi 4: Cộng tổng toàn cây chia số tầng, hoặc trung bình của trung bình
Trung bình phải tính **riêng từng tầng**.

In [16]:
def wrong_no_size(root):
    queue, result = deque([root]), []
    while queue:
        total = count = 0
        while queue:                          # SAI: ăn cả tầng sau
            node = queue.popleft()
            total += node.val
            count += 1
            queue.extend(c for c in (node.left, node.right) if c)
        result.append(total / count)
    return result

def wrong_floor_div(root):
    queue, result = deque([root]), []
    while queue:
        size, total = len(queue), 0
        for _ in range(size):
            node = queue.popleft()
            total += node.val
            queue.extend(c for c in (node.left, node.right) if c)
        result.append(total // size)          # SAI: chia nguyên
    return result

def wrong_while_len(root):
    queue, result = deque([root]), []
    while queue:
        i, total, cnt = 0, 0, 0
        level = list(queue)
        queue.clear()
        while i < len(level):                 # SAI: level dài thêm khi ta thêm con vào chính nó
            node = level[i]
            total += node.val
            cnt += 1
            level.extend(c for c in (node.left, node.right) if c)
            i += 1
        result.append(total / cnt)
    return result

t = build_tree(EX1)
print("Không chụp size :", wrong_no_size(t))
print("Chia //         :", wrong_floor_div(t))
print("while < len     :", wrong_while_len(t))
print("Đúng            :", average_levels_bfs(t))

Không chụp size : [10.8]
Chia //         : [3, 14, 11]
while < len     : [10.8]
Đúng            : [3.0, 14.5, 11.0]


### Lỗi 5: Thêm `None` vào queue
`queue.append(node.left)` khi `node.left` là `None` → vòng sau `None.val` lỗi `AttributeError`. Luôn kiểm tra `if node.left:` trước khi thêm.

## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu "theo tầng" (level order):** đề nói "each level", "row", "depth", "level by level", "right side view", "zigzag". → BFS + chụp `size` (hoặc DFS mang `depth`).

**Khung BFS theo tầng:**
```python
queue = deque([root])
while queue:
    size = len(queue)
    for _ in range(size):
        node = queue.popleft()
        ... xử lý node ...
        thêm con (nếu có)
    ... tổng kết tầng ...
```

**Bài cùng khung:** 102 (Level Order), 103 (Zigzag), 107 (Level Order II), 199 (Right Side View), 515 (Largest Value in Each Row), 1161 (Maximum Level Sum), 429 (N-ary Level Order).

Demo một biến thể cùng khung – **Right Side View (LC 199)**: phần tử **cuối cùng** lấy ra ở mỗi tầng là node nhìn thấy từ bên phải.

In [17]:
def right_side_view(root):
    if root is None:
        return []
    queue, view = deque([root]), []
    while queue:
        size = len(queue)
        for i in range(size):
            node = queue.popleft()
            if i == size - 1:              # node cuối của tầng
                view.append(node.val)
            queue.extend(c for c in (node.left, node.right) if c)
    return view

print(right_side_view(build_tree(EX1)))    # [3, 20, 7]
print(right_side_view(build_tree(EX2)))    # [3, 20, 7]

[3, 20, 7]
[3, 20, 7]


## 15. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Tính tay.** `root = [1,2,3,4,5,null,6]`: trung bình từng tầng?

**Bài 2 — Đoán output.** `print(7 / 2, 7 // 2, -7 // 2)`.

**Bài 3 — Viết lại** `averageOfLevels` bằng BFS với `size`.

**Bài 4 — Largest Value in Each Row (LC 515):** giá trị lớn nhất mỗi tầng.

**Bài 5 — Maximum Level Sum (LC 1161):** trả về **số thứ tự tầng** (tính từ **1**) có tổng lớn nhất; hòa thì lấy tầng nhỏ nhất.

**Bài 6 — Mini challenge (Zigzag, LC 103):** trả về list các tầng, tầng chẵn trái → phải, tầng lẻ phải → trái.

In [18]:
answer_ex1 = None    # TODO: list float
answer_ex2 = None    # TODO: tuple 3 giá trị


def average_levels_exercise(root):
    # TODO 1: queue = deque([root])
    # TODO 2: mỗi tầng: size = len(queue), popleft đúng size lần
    # TODO 3: append tổng / size
    pass


def largest_values(root):
    # TODO: như BFS nhưng giữ max thay vì tổng
    pass

# Kiểm tra sau khi làm:
# assert average_levels_exercise(build_tree([3, 9, 20, None, None, 15, 7])) == [3.0, 14.5, 11.0]
# assert largest_values(build_tree([1, 3, 2, 5, 3, None, 9])) == [1, 3, 9]

In [19]:
def max_level_sum(root):
    # TODO: trả về tầng (1-based) có tổng lớn nhất, hòa lấy tầng nhỏ
    pass


def zigzag_level_order(root):
    # TODO: BFS theo tầng, đảo list tầng ở các tầng lẻ
    pass

# Kiểm tra sau khi làm:
# assert max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])) == 2
# assert zigzag_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [20, 9], [15, 7]]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Vẽ cây theo tầng. (2) Tầng 1: `2, 3`; tầng 2: `4, 5, 6`. (3) `[1, 2.5, 5]`.

**Bài 2:** (1) `/` ra float. (2) `//` làm tròn **xuống** (về âm vô cực). (3) `-7 // 2` là `-4`, không phải `-3`.

**Bài 3:** (1) Khung BFS mục 14. (2) `level_sum / size`. (3) Xem mục 5.

**Bài 4:** (1) Khung BFS. (2) Khởi tạo `best = -math.inf` mỗi tầng. (3) `best = max(best, node.val)`.

**Bài 5:** (1) Khung BFS, đếm tầng `level` từ 1. (2) So sánh tổng tầng với `best_sum` bằng `>` (không `>=`) để giữ tầng nhỏ khi hòa. (3) Trả `best_level`.

**Bài 6:** (1) Khung BFS, gom `vals` của tầng. (2) Nếu tầng lẻ: `vals.reverse()` (hoặc `vals[::-1]`). (3) Vẫn thêm con theo thứ tự trái → phải; chỉ đảo **kết quả** của tầng.

## 17. Đáp án tham khảo (chạy kiểm tra)

In [20]:
ans_ex1 = average_levels_bfs(build_tree([1, 2, 3, 4, 5, None, 6]))   # [1.0, 2.5, 5.0]
ans_ex2 = (7 / 2, 7 // 2, -7 // 2)                                  # (3.5, 3, -4)


def ans_average_levels(root):
    queue, result = deque([root]), []
    while queue:
        size, total = len(queue), 0
        for _ in range(size):
            node = queue.popleft()
            total += node.val
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(total / size)
    return result


def ans_largest_values(root):
    if root is None:
        return []
    queue, result = deque([root]), []
    while queue:
        best = -math.inf
        for _ in range(len(queue)):
            node = queue.popleft()
            best = max(best, node.val)
            queue.extend(c for c in (node.left, node.right) if c)
        result.append(best)
    return result


def ans_max_level_sum(root):
    queue = deque([root])
    level, best_level, best_sum = 0, 0, -math.inf
    while queue:
        level += 1
        total = 0
        for _ in range(len(queue)):
            node = queue.popleft()
            total += node.val
            queue.extend(c for c in (node.left, node.right) if c)
        if total > best_sum:
            best_sum, best_level = total, level
    return best_level


def ans_zigzag_level_order(root):
    if root is None:
        return []
    queue, result, level = deque([root]), [], 0
    while queue:
        vals = []
        for _ in range(len(queue)):
            node = queue.popleft()
            vals.append(node.val)
            queue.extend(c for c in (node.left, node.right) if c)
        result.append(vals[::-1] if level % 2 == 1 else vals)
        level += 1
    return result


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert close_lists(ans_ex1, [1.0, 2.5, 5.0]) and ans_ex2 == (3.5, 3, -4)
assert ans_average_levels(build_tree([3, 9, 20, None, None, 15, 7])) == [3.0, 14.5, 11.0]
assert ans_largest_values(build_tree([1, 3, 2, 5, 3, None, 9])) == [1, 3, 9]
assert ans_max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])) == 2
assert ans_max_level_sum(build_tree([989, None, 10250, 98693, -89388, None, None, None, -32127])) == 2
assert ans_zigzag_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [20, 9], [15, 7]]
assert ans_zigzag_level_order(build_tree([1, 2, 3, 4, None, None, 5])) == [[1], [3, 2], [4, 5]]
print("Đáp án: tất cả test PASS")

Bài 1: [1.0, 2.5, 5.0] | Bài 2: (3.5, 3, -4)
Đáp án: tất cả test PASS


## 18. Template nhớ nhanh và checklist submit

```python
from collections import deque

class Solution:
    def averageOfLevels(self, root):
        result, queue = [], deque([root])
        while queue:
            size, level_sum = len(queue), 0          # CHỤP size đầu tầng
            for _ in range(size):
                node = queue.popleft()
                level_sum += node.val
                if node.left:  queue.append(node.left)
                if node.right: queue.append(node.right)
            result.append(level_sum / size)          # chia bằng /
        return result
```

Câu nhớ: **"chụp size – lấy đúng size node – tổng kết tầng"**.

### Checklist
- [ ] Chụp `size = len(queue)` **trước** khi lấy node của tầng?
- [ ] Dùng `/` (float), không `//`?
- [ ] Chỉ thêm con khác `None` vào queue?
- [ ] Dùng `deque.popleft()` (O(1)), không `list.pop(0)`?
- [ ] Nghĩ tới tràn số nếu viết bằng ngôn ngữ khác (tổng tầng có thể > `2^31`)?